# Imports and preprocessing

In [79]:
import csv
import numpy as np
from pathlib import Path
import sys, os
import json
import importlib
sys.path.append(os.path.join(os.getcwd(), '..'))

from utils import eda_helpers, all_functions, cv, sampling_methods
import implementations

importlib.reload(eda_helpers)
importlib.reload(all_functions)
importlib.reload(cv)
importlib.reload(sampling_methods)
importlib.reload(implementations)

from utils.eda_helpers import clean_data, read_csv_numeric
from utils.all_functions import sigmoid
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    weighted_logistic_regression,
)

file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')

In [2]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))

In [80]:
final_feature_set = ['_CASTHM1', 'FLUSHOT6', 'METVL11_', '_PASTRNG', 'EDUCA', 'CHCCOPD1', 'DRADVISE', 
                     'CHCKIDNY', 'ARTHSOCL', 'MENTHLTH', 'SMOKE100', 'DOCTDIAB', 'DIABAGE2', 'VETERAN3', 
                     'CHKHEMO3', 'SEX', '_DRNKWEK', 'JOINPAIN', 'MARITAL', 'EXERANY2', '_PACAT1', 'INSULIN', 
                     'DIFFALON', 'FRUIT1', 'QLACTLM2', 'BLIND', 'PADUR2_', 'PAFREQ2_', 'LMTJOIN3', 'CHOLCHK', 
                     'POORHLTH', 'DIABETE3', '_FRUTSUM', 'PERSDOC2', 'EXRACT21', 'FEETCHK', '_VEGESUM', 'PNEUVAC3', 
                     'BPHIGH4', 'GENHLTH', 'EXRACT11', '_BMI5', 'DECIDE', '_ASTHMS1', '_MRACE1', '_LMTSCL1', 'ORNGDAY_', 
                     'MSCODE', 'PHYSHLTH', 'TOLDHI2', 'PADUR1_', 'DIFFWALK', 'BEANDAY_', 'CHCSCNCR', 'PAFREQ1_', '_AIDTST3', 
                     'FLSHTMY2', 'PA1VIGM_', 'METVL21_', 'SXORIENT', 'ARTHDIS2', 'PA1MIN_', 'MAXVO2_', '_RACE_G1', 'LONGWTCH', 
                     'SCNTWRK1', 'FEETCHK2', 'EMPLOY1', 'BLOODCHO', 'USENOW3', 'DRNKANY5', '_PAREC1', 'DIFFDRES', 'USEEQUIP', 
                     'ADDEPEV2', '_AGE80', 'HTM4', 'INCOME2', 'CVDSTRK3', 'CHCOCNCR', 'MEDCOST', 'PREDIAB1', 'BPMEDS', 'HAVARTH3', 
                     'CHECKUP1', 'TRNSGNDR', 'CHILDREN']

In [81]:
del x_indices["Id"]
x_data = clean_data(x_data, x_indices)

KeyError: 'Id'

In [7]:
# --- Keep only the columns of interest ---
x_data_interested = x_data[:, [x_indices[col] for col in final_feature_set]]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Drop columns that are all NaN ---
valid_cols = ~np.isnan(x_data_interested).all(axis=0)
x_data_interested = x_data_interested[:, valid_cols]
interesting_columns = [col for col, keep in zip(final_feature_set, valid_cols) if keep]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Impute remaining NaNs with column mean ---
col_mean = np.nanmean(x_data_interested, axis=0)
inds = np.where(np.isnan(x_data_interested))
x_data_interested[inds] = np.take(col_mean, inds[1])

# --- Standardize once ---
x_data_interested = (x_data_interested - np.mean(x_data_interested, axis=0)) / np.std(x_data_interested, axis=0)

# --- Prepare y ---
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.flatten()

# --- Checks ---
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data_interested)))
print("Min std across features:", np.min(np.std(x_data_interested, axis=0)))
print("Label distribution (mean):", np.mean(y))

Unique labels: [0. 1.]
Any NaN in features? False
Min std across features: 0.9999999999999998
Label distribution (mean): 0.08830207079403295


# Evaluation

In [82]:
from utils.cv import (
    cross_validate,
    train_logreg,
    predict,
    compute_training_error,
)
from utils.sampling_methods import (
    undersample,
    oversample,
    smote
)


In [84]:
sampling_methods = {
    "none": None,
    "undersample": undersample,
    "oversample": oversample,
    "smote": smote
}
sampling_kwargs = {
    "undersample": {"ratio": 2.0, "seed": 42},
    "oversample": {"ratio": 3.0, "seed": 42},
    "smote": {"k": 5, "ratio": 3.0, "seed": 42, "batch_size": 20},
}

# all_results = []

In [ ]:
# --- Cross-validation for all logistic regression variants ---

for kind in ["w_logistic"]:
    print(f"\n=== MODEL: {kind.upper()} ===")

    for s_name, s_func in sampling_methods.items():
        print(f"\n--- Sampling: {s_name.upper()} ---")

        # cross-validation
        results = cross_validate(
            model_fn=lambda y, X: train_logreg(
                y, X,
                kind=kind,
                gamma=0.01,
                lambda_ = 0.01,
                max_iters=500,
                verbose=True
            ),
            X=x_data_interested,
            y=y,
            k_folds=5,
            seed=42,
            sampling_fn=s_func,
            sampling_kwargs=sampling_kwargs.get(s_name, None),
            predict_fn=lambda X, w: (predict(X, w), predict(X, w, return_prob=True)),
            metric_fn=compute_training_error,
            thresholds = 0.75,
            use_predictions=False,
            verbose=True
        )

        # store results
        model_result = {"model": kind, "sampling": s_name, **results}
        all_results.append(model_result)
        print("Average CV metrics:", results)



=== MODEL: W_LOGISTIC ===

--- Sampling: NONE ---
[w_logistic] Training finished — final loss: 0.522401
Fold 1/5 @ thr=0.75 → error:0.151, accuracy:0.849, precision:0.311, recall:0.605, f1:0.411, roc_auc:0.147
[w_logistic] Training finished — final loss: 0.522206
Fold 2/5 @ thr=0.75 → error:0.150, accuracy:0.850, precision:0.315, recall:0.590, f1:0.410, roc_auc:0.148
[w_logistic] Training finished — final loss: 0.523364
Fold 3/5 @ thr=0.75 → error:0.148, accuracy:0.852, precision:0.321, recall:0.601, f1:0.419, roc_auc:0.143
[w_logistic] Training finished — final loss: 0.523424
Fold 4/5 @ thr=0.75 → error:0.150, accuracy:0.850, precision:0.315, recall:0.605, f1:0.414, roc_auc:0.143
[w_logistic] Training finished — final loss: 0.522903
Fold 5/5 @ thr=0.75 → error:0.152, accuracy:0.848, precision:0.314, recall:0.601, f1:0.413, roc_auc:0.145
Average CV metrics: {0.75: {'error': np.float64(0.15043808188702953), 'accuracy': np.float64(0.8495619181129704), 'precision': np.float64(0.315303291

In [ ]:
output_dir = os.path.join(os.getcwd(), '..', 'results', 'eval')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'all_results.json')
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)